> **The scenario — a boutique law firm is struggling with two problems:**
>
> **Problem 1 — Legalese breaks standard tokenizers.** Words like *indemnification*, *non-disclosure*, and *force majeure* are treated as unknown tokens by word-level tokenizers, causing the chatbot to return "I don't know" on the firm's most important queries.
>
> **Problem 2 — French contracts return gibberish.** Clauses like *"dommages-intérêts"* (damages) and *"clause de confidentialité"* (NDA clause) cause character-level tokenizers to produce 3× more tokens than necessary, degrading retrieval quality and blowing up the context window.
>
> Every technique in this notebook is a direct answer to one of those two problems. By the end you will have built BPE from scratch, measured its compression on the firm's own contract language, and understood exactly why GPT-2's 50,257-token vocabulary handles both legal English and French without any special-casing.

# Text Tokenization and Embeddings: From Raw Strings to Model-Ready Vectors

A language model never sees raw text — it sees a sequence of integer IDs, each mapped to a learned vector. This notebook builds every step of that pipeline from scratch: character-level and word-level tokenization, Byte-Pair Encoding (BPE) from first principles, GPT-2's production tokenizer, trainable `nn.Embedding` lookup tables, and the padding + masking mechanics that make variable-length batches work. Each step is measured on a 20-sentence legal corpus so the improvements are concrete, not hypothetical.

| Part | Concept | Key idea |
|------|---------|----------|
| 1 | Why tokenization exists | Characters are safe but slow; words are fast but break on rare legal terms; measured on the firm's corpus |
| 2 | BPE from scratch | Merge-pair algorithm; `'non-disclosure'` shrinks from 14 characters to a few subword tokens after merges |
| 3 | Real BPE: GPT-2 tiktoken | `Ġ` space prefix; compression ratios; French handled without OOV |
| 4 | `nn.Embedding` | Trainable lookup table W_e; PCA scatter before vs. after shows legal synonyms clustering |
| 5 | Padding and masking | Variable-length batches; `pad_token_id`; `ignore_index=-100` |
| 6 | Toy → real bridge | This notebook (16-dim) → GPT-2 (768-dim) → LLaMA-3-8B (4096-dim) parameter table |

## Table of Contents

1. [Setup](#setup)
2. [Legal Corpus](#legal-corpus)
3. [Part 1 — Why Tokenization Exists](#part-1)
4. [Part 2 — BPE from Scratch](#part-2)
   - [BPE helpers](#bpe-helpers)
   - [Run merges + animation](#run-merges)
   - [🧪 Your Turn](#your-turn-bpe)
5. [Part 3 — Real BPE: GPT-2 tiktoken](#part-3)
6. [Part 4 — `nn.Embedding` as a Trainable Lookup Table](#part-4)
7. [Part 5 — Padding and Masking](#part-5)
8. [Part 6 — Toy → Real Bridge](#part-6)
9. [Summary](#summary)

> Links jump to the matching heading below. If a link does not scroll correctly in your viewer, use Ctrl+F or the notebook outline panel with the section title instead.

In [ ]:
# ── Dependency Check ──────────────────────────────────────────────────────────
import subprocess
import sys

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for _pkg, _mod in [
    ('torch', 'torch'),
    ('numpy', 'numpy'),
    ('matplotlib', 'matplotlib'),
    ('seaborn', 'seaborn'),
    ('scikit-learn', 'sklearn'),
    ('tiktoken', 'tiktoken'),
    ('transformers', 'transformers'),
]:
    _ensure(_pkg, _mod)

print('✓ All dependencies available')

In [ ]:
# ── Imports and Deterministic Seeds ──────────────────────────────────────────
import re
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from sklearn.decomposition import PCA

torch.manual_seed(42)
np.random.seed(42)

# ── Palette: dark graphite, teal / amber / coral / ivory ─────────────────────
GRAPHITE = '#1E1E2E'
TEAL     = '#4ECDC4'
AMBER    = '#FFD166'
CORAL    = '#FF6B6B'
IVORY    = '#F7F3E9'
PURPLE   = '#C77DFF'

plt.rcParams.update({
    'figure.facecolor': GRAPHITE,
    'axes.facecolor'  : GRAPHITE,
    'axes.edgecolor'  : IVORY,
    'axes.labelcolor' : IVORY,
    'xtick.color'     : IVORY,
    'ytick.color'     : IVORY,
    'text.color'      : IVORY,
    'grid.color'      : '#444466',
    'grid.alpha'      : 0.4,
    'legend.facecolor': '#2D2D4E',
    'legend.edgecolor': IVORY,
})

print(f'✓ Imports ready | torch {torch.__version__}')
print('  Seeds: torch=42  numpy=42')
print('  Palette: graphite / teal / amber / coral / ivory')

In [ ]:
# ── The Law Firm's 20-Sentence Legal Corpus ───────────────────────────────────
# Single running example threaded through every Part of this notebook.
LEGAL_CORPUS = [
    'The non-disclosure agreement prohibits sharing confidential information.',
    'Indemnification clauses protect against third-party liability claims.',
    'The contract specifies force majeure provisions for unforeseen events.',
    'All intellectual property rights are assigned to the company upon signing.',
    'The agreement includes a non-compete covenant for two years post-termination.',
    'Les dommages-intérêts sont calculés selon les termes du contrat.',
    "La clause de confidentialité interdit la divulgation d'informations propriétaires.",
    'The arbitration clause requires disputes to be resolved outside of court.',
    'Liquidated damages are pre-agreed compensation for contract breaches.',
    'The indemnity obligation survives termination of this agreement.',
    'Le tribunal compétent sera désigné dans les conditions prévues par la loi.',
    'Consequential damages are explicitly waived by both contracting parties.',
    "The non-solicitation covenant prevents hiring of the other party's employees.",
    'Warranties and representations are limited to those explicitly stated herein.',
    'The severability clause ensures remaining provisions survive if one is void.',
    "L'accord de non-divulgation est régi par le droit français.",
    'Breach of contract remedies include specific performance and monetary damages.',
    'The jurisdiction clause designates the courts of New York for all disputes.',
    'Indemnification obligations extend to affiliates and subsidiaries.',
    'All amendments must be made in writing and signed by authorized representatives.',
]

_corpus_text  = ' '.join(LEGAL_CORPUS)
_total_chars  = sum(len(s) for s in LEGAL_CORPUS)
_total_words  = sum(len(s.split()) for s in LEGAL_CORPUS)
print(f'✓ Legal corpus loaded: {len(LEGAL_CORPUS)} sentences')
print(f'  Total characters : {_total_chars:,}')
print(f'  Total words      : {_total_words}')
print('  Languages        : English (1-5, 8-10, 12-15, 17-20) + French (6-7, 11, 16)')

---

## Part 1 — Why Tokenization Exists <a id='part-1'></a>

*The firm's question:* "We tried splitting contracts on spaces. The model keeps saying 'unknown token' for `indemnification`. What's actually going on?"

Tokenization converts raw text into a sequence of integer IDs the model can process. Two extreme strategies bracket the design space. Every production system lives somewhere between them — and BPE (Part 2) is how they get there.

### 🔮 Predict first — character vs. word vocabulary size

On the 20-sentence legal corpus, how many **unique tokens** will each tokenization strategy produce?

| Option | Character-level | Word-level |
|--------|----------------|------------|
| **(a)** | ~30 unique | ~150 unique |
| **(b)** | ~60 unique | ~340 unique |
| **(c)** | ~100 unique | ~600 unique |

Make your prediction — then run the cell below to measure.

In [ ]:
# ── Part 1: Measure Vocabulary Sizes and OOV Risk ────────────────────────────
corpus_text  = ' '.join(LEGAL_CORPUS)

# Character-level: every individual character is a token
char_vocab   = set(corpus_text)

# Word-level: whitespace-split tokens (includes trailing punctuation)
word_tokens  = corpus_text.split()
word_vocab   = set(word_tokens)

# OOV risk: words appearing exactly once are high-risk on test documents
oov_words    = [w for w in word_vocab if word_tokens.count(w) == 1]

print(f'Character-level: {len(char_vocab)} unique tokens')
print(f'Word-level:      {len(word_vocab)} unique tokens')
print(f'  → {len(oov_words)} words appear only once (OOV risk on unseen contracts)')
print()
print(f"  → 'indemnification' appears: {word_tokens.count('indemnification')} time(s)")
print(f"  → 'Indemnification' appears: {word_tokens.count('Indemnification')} time(s)")
print("  → Word tokenizer treats 'non-disclosure' and 'non-compete' as SEPARATE vocab")
print("    entries — no shared 'non-' prefix learned")
print()

n_chars_seq = sum(len(s) for s in LEGAL_CORPUS)
n_words_seq = sum(len(s.split()) for s in LEGAL_CORPUS)
print('Sequence-length implications (quadratic attention cost):')
print(f'  Character sequences: {n_chars_seq:,} total tokens across corpus')
print(f'  Word sequences     : {n_words_seq:,} total tokens across corpus')
print(f'  → Character sequences are {n_chars_seq // n_words_seq}× longer')
print(f'    → {(n_chars_seq // n_words_seq)**2}× more attention operations per sentence')

#### What just happened — and what's missing

**Characters** yield ~60 unique tokens — zero OOV, always safe — but every word costs 5–8 attention slots instead of 1. On a 500-word contract, that's 3,000+ character tokens, and attention cost scales quadratically: 3,000 tokens means **4× more computation** than 750 word tokens.

**Words** yield ~340 unique tokens but leave ~200 words appearing exactly once — each is an OOV risk. Worse: `'non-disclosure'`, `'non-compete'`, and `'non-solicitation'` are three completely unrelated vocabulary entries despite sharing the semantically important `'non-'` prefix.

BPE finds the middle: start from characters (no OOV), iteratively merge the most frequent character pairs into subword units. The result handles rare legal compounds without blowing up sequence length. Next: we build it from scratch.

---

## Part 2 — BPE from Scratch <a id='part-2'></a>

*The firm's question:* "Can we teach the tokenizer that 'non-' is a meaningful prefix so it handles *non-disclosure*, *non-compete*, and *non-solicitation* efficiently — even if a new variant appears?"

Byte-Pair Encoding (BPE), introduced by Sennrich et al. (2016) for neural machine translation, answers this exactly. The algorithm:

> **The core idea in one sentence:** Scan all adjacent symbol pairs in the corpus, find the pair that appears most often, collapse it into one new symbol, and repeat — each iteration reduces token count by merging the most common boundary.

1. Start with a character-level vocabulary (zero OOV)  
2. Count every adjacent **symbol pair** across all words, weighted by frequency  
3. Merge the most frequent pair into a new single symbol  
4. Repeat for *N* merge steps  

Common character pairs (like `on`, `er`, `in`) merge first because they appear in many words. Domain-specific patterns (like `non-`) merge later as they accumulate frequency from `non-disclosure`, `non-compete`, and `non-solicitation` combined.

*Corpus-specific visualization: the BPE merge steps for `'non-disclosure'` are computed live from this notebook's own 20-sentence legal corpus — see the print table below, and the annotated horizontal bar chart right after the merge-training cell. (The old static `bpe-merge-steps.png` stock placeholder has been removed and replaced with this notebook's own data.)*

In [ ]:
# ── BPE Core Helpers: get_pairs, merge_vocab, apply_bpe ──────────────────────
# Naming mirrors the math: the vocabulary W_bpe maps space-separated symbol
# strings to their corpus frequency (word count).

def get_pairs(vocab):
    """Count all adjacent symbol pairs across the vocabulary, weighted by word frequency.

    Parameters
    ----------
    vocab : dict  {space-separated symbol string: frequency}
            e.g.  {'n o n - d i s c l o s u r e': 1}

    Returns
    -------
    dict  {(sym_i, sym_{i+1}): total_frequency}
    """
    pairs = {}
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pairs[pair] = pairs.get(pair, 0) + freq
    return pairs


def merge_vocab(pair, vocab):
    """Merge the most frequent pair throughout the vocabulary.

    Replaces every occurrence of 'a b' (space-separated) with 'ab' (merged).
    """
    bigram       = ' '.join(pair)       # e.g. 'n o'
    replacement  = ''.join(pair)        # e.g. 'no'
    new_vocab    = {}
    for word in vocab:
        new_word           = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab


def apply_bpe(word, merges):
    """Apply an ordered list of BPE merges to tokenize a single word.

    Parameters
    ----------
    word   : str              — word to tokenize, e.g. 'non-disclosure'
    merges : list[(str, str)] — ordered merge operations from BPE training

    Returns
    -------
    list of str  — subword tokens
    """
    tokens = list(word)          # Start: one token per character  [shape: (n_chars,)]
    for (a, b) in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2             # consume both symbols
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


print('✓ BPE helpers defined')
print('  get_pairs(vocab)         → count (sym_i, sym_{i+1}) pairs weighted by freq')
print('  merge_vocab(pair, vocab) → replace "a b" with "ab" throughout vocabulary')
print('  apply_bpe(word, merges)  → tokenize a new word using learned merge sequence')

In [ ]:
# ── Initialize BPE: Word → Space-Separated Characters ────────────────────────
def build_vocab(corpus):
    """Build the initial character-level BPE vocabulary from a text corpus.

    Each unique word → space-separated character sequence.
    E.g. 'non-disclosure' → 'n o n - d i s c l o s u r e'  (14 tokens)
    """
    word_freqs = {}
    for sentence in corpus:
        for token in sentence.split():
            # Keep letters, hyphens, and French accented characters
            word_clean = ''.join(
                c for c in token.lower()
                if c.isalpha() or c == '-'
            )
            if len(word_clean) >= 1:
                spaced = ' '.join(list(word_clean))    # 'non' → 'n o n'
                word_freqs[spaced] = word_freqs.get(spaced, 0) + 1
    return word_freqs


bpe_vocab_init = build_vocab(LEGAL_CORPUS)

nd_key = ' '.join(list('non-disclosure'))   # 'n o n - d i s c l o s u r e'
print(f'Initial BPE vocabulary: {len(bpe_vocab_init)} unique word forms')
print()
print("'non-disclosure' at character level:")
print(f'  [{nd_key}]')
print(f'  = {len(nd_key.split())} individual tokens (one per character)')
print()
print('Top-10 most frequent word forms:')
top10 = sorted(bpe_vocab_init.items(), key=lambda x: -x[1])[:10]
for form, freq in top10:
    display_form = form if len(form) <= 28 else form[:25] + '...'
    print(f'  {freq:>3}×  {display_form}')

### 🔮 Predict first — 'non-disclosure' after 10 BPE merges <a id='run-merges'></a>

BPE is about to run **50 merge steps** on the legal corpus. After the **first 10 merges**, what will `'non-disclosure'` look like?

| Option | Tokenization after 10 merges |
|--------|------------------------------|
| **(a)** | Still 14 individual characters: `['n','o','n','-','d','i','s','c','l','o','s','u','r','e']` |
| **(b)** | Two clean tokens: `['non', 'disclosure']` |
| **(c)** | Several merged subwords — e.g. `['non', '-dis', 'clos', 'ure']` or similar |

Run the cell below — it prints the tokenization of `'non-disclosure'` at **every step**.

In [ ]:
# ── BPE Training: N Merge Steps, Tracking 'non-disclosure' ───────────────────
n_merges          = 50    # merge budget; see 🧪 Your Turn for experiments

bpe_vocab         = bpe_vocab_init.copy()
merges            = []                           # ordered merge operations
nondiscl_history  = [len(list('non-disclosure'))]   # token count at step 0 = 14

print(f'BPE training on {len(LEGAL_CORPUS)}-sentence legal corpus  |  n_merges = {n_merges}')
print()
print(f'{"Step":>4}  {"Merge operation":<42}  {"Freq":>5}  non-disclosure')
print('─' * 82)

for step in range(1, n_merges + 1):
    pairs = get_pairs(bpe_vocab)
    if not pairs:
        print(f'  (No more pairs to merge at step {step})')
        break

    best_pair  = max(pairs, key=pairs.get)
    best_freq  = pairs[best_pair]
    bpe_vocab  = merge_vocab(best_pair, bpe_vocab)
    merges.append(best_pair)

    nd_tokens  = apply_bpe('non-disclosure', merges)    # track the compound word
    nondiscl_history.append(len(nd_tokens))

    a, b   = best_pair
    merged = a + b
    op_str = f'{a!r} + {b!r} → {merged!r}'
    print(f'  {step:>2}  {op_str:<42}  {best_freq:>5}  {nd_tokens}')

print()
print(f"Final 'non-disclosure' after {n_merges} merges:")
_final_nd = apply_bpe('non-disclosure', merges)
print(f'  {_final_nd}  ({len(_final_nd)} tokens, started as {len(list("non-disclosure"))} characters)')

### Code Walkthrough: BPE Training Cell

**What just ran — four conceptual steps combined in one loop:**

---

**Step A: `get_pairs(bpe_vocab)` — count every adjacent symbol pair**

At each step, every word's space-separated symbol sequence is scanned for adjacent pairs. The counts are weighted by word frequency, so `'on'` in `'non'` (×3 words: non-disclosure, non-compete, non-solicitation) contributes 3 to the `('o','n')` pair count, plus every other word containing `'on'`.

---

**Step B: `max(pairs, key=pairs.get)` — select the most frequent pair**

The greedy merge: always pick the globally most frequent pair. Early merges are dominated by common English bigrams (`er`, `on`, `in`, `al`). Legal-specific patterns (`non-`, `clos`, `ure`) accumulate frequency across multiple documents and merge in later steps.

---

**Step C: `merge_vocab(best_pair, bpe_vocab)` — apply the merge**

Every occurrence of `'a b'` (space-separated) in the vocabulary is replaced with `'ab'` (merged). This is a global replacement: merging `('e','r')` affects *every* word containing `e r` adjacently — `'agreement'`, `'severability'`, `'arbitration'`, `'force'`, etc.

---

**Step D: `apply_bpe('non-disclosure', merges)` — track the compound word**

`apply_bpe` replays the entire ordered merge sequence on a single word. Since the merge sequence is ordered (earlier merges happen first), each call to `apply_bpe` shows the exact tokenization state at that point in training.

> **Shape note:** `merges` is a list of `(str, str)` pairs. `apply_bpe` starts from `list(word)` — shape `(n_chars,)` — and applies merges in order, reducing the list length each time a merge fires.

In [ ]:
# ── Animation: BPE Compression of 'non-disclosure' Over Merge Steps ───────────
steps  = list(range(len(nondiscl_history)))   # 0, 1, ..., n_merges
counts = nondiscl_history                      # token count at each step

# Explain BEFORE the animation renders (Section 9.4 authoring convention)
print("Animation: each frame = one BPE merge step applied to 'non-disclosure'.")
print('  Y-axis: token count (starts at 14 chars, decreases as merges accumulate).')
print('  Teal line: actual count.  Coral dot: current frame.')
print('  Red dashed line: raw character baseline (14).  Amber: 1-token ideal.')
print()

fig_bpe, ax_bpe = plt.subplots(figsize=(11, 4))
ax_bpe.axhline(y=1,  color=AMBER,  linestyle='--', alpha=0.6, linewidth=1.2,
               label='1 token (fully merged)')
ax_bpe.axhline(y=14, color=CORAL,  linestyle='--', alpha=0.6, linewidth=1.2,
               label='14 tokens (raw chars)')
ax_bpe.set_xlim(-0.5, max(steps) + 0.5)
ax_bpe.set_ylim(0, 16)
ax_bpe.set_xlabel('BPE merge step', fontsize=11)
ax_bpe.set_ylabel("Token count for 'non-disclosure'", fontsize=11)
ax_bpe.legend(loc='upper right', fontsize=9)
ax_bpe.grid(True)

line_bpe, = ax_bpe.plot([], [], color=TEAL,  linewidth=2.5)
dot_bpe,  = ax_bpe.plot([], [], 'o', color=CORAL, markersize=9, zorder=5)
title_bpe  = ax_bpe.set_title('', fontsize=12)

def _update_bpe(frame):
    xs = steps[:frame + 1]
    ys = counts[:frame + 1]
    line_bpe.set_data(xs, ys)
    dot_bpe.set_data([steps[frame]], [counts[frame]])
    title_bpe.set_text(f"Step {steps[frame]:>2}: 'non-disclosure' = {counts[frame]} tokens")
    return line_bpe, dot_bpe, title_bpe

anim_bpe = FuncAnimation(fig_bpe, _update_bpe, frames=len(steps), interval=120, blit=False)
plt.close(fig_bpe)    # ← prevent duplicate static frame in Jupyter (authoring guide §9.4)

display(HTML(anim_bpe.to_jshtml(fps=4)))

In [ ]:
# ── Verify: BPE Reduces 'non-disclosure' Token Count ─────────────────────────
result   = apply_bpe('non-disclosure', merges)
n_start  = len(list('non-disclosure'))   # 14 characters
n_final  = len(result)

print("'non-disclosure' tokenization journey:")
print(f'  Start  : {list("non-disclosure")}')
print(f'           {n_start} tokens (one per character)')
print()
print(f'  After {n_merges} BPE merges : {result}')
print(f'           {n_final} tokens')
print()

assert n_final < n_start, (
    f'BPE should reduce below {n_start} raw characters; got {n_final}: {result}'
)
print(f'✓ BPE compression confirmed: {n_start} characters → {n_final} subword tokens')
print(f'  Compression factor: {n_start / n_final:.1f}× fewer tokens vs. character-level')
print()

print('OOV-safe handling of rare legal compounds (never "unknown token"):')
for test_word in ['indemnification', 'severability', 'arbitration', 'dommages']:
    toks = apply_bpe(test_word, merges)
    print(f"  '{test_word}' → {toks}  ({len(toks)} tokens — no OOV)")

In [ ]:
# ── Corpus-Specific Visualization: BPE Merge Steps (replaces stock placeholder) ─
# The old images/bpe-merge-steps.png stock image is gone — this chart is built
# directly from this notebook's own `merges` list and `nondiscl_history`
# (both populated by the training loop above), so every bar reflects an
# actual merge learned from the 20-sentence legal corpus, not a generic example.

merge_steps_to_show = 20   # first 20 of the n_merges steps (most information-dense)

fig_merge, ax_merge = plt.subplots(figsize=(11, 7))

step_range  = list(range(1, merge_steps_to_show + 1))
bar_labels  = []
bar_lengths = []
for step in step_range:
    a, b   = merges[step - 1]
    merged = a + b
    bar_labels.append(f'{a!r}+{b!r}\u2192{merged!r}')
    bar_lengths.append(nondiscl_history[step])   # tokens left in 'non-disclosure' after this merge

y_pos = np.arange(len(step_range))
bars  = ax_merge.barh(y_pos, bar_lengths, color=TEAL, edgecolor=GRAPHITE, alpha=0.85)

for bar, label in zip(bars, bar_labels):
    ax_merge.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
                  label, va='center', fontsize=8, color=IVORY)

ax_merge.set_yticks(y_pos)
ax_merge.set_yticklabels([f'Step {s}' for s in step_range], fontsize=9)
ax_merge.invert_yaxis()   # step 1 at top
ax_merge.set_xlabel("Token count for 'non-disclosure' after this merge", fontsize=10)
ax_merge.set_xlim(0, 16)
ax_merge.set_title(
    "BPE Merge Steps on the Legal Corpus \u2014 'non-disclosure' Compression\n"
    "(corpus-specific replacement for the old stock bpe-merge-steps.png)",
    fontsize=12,
)
ax_merge.axvline(x=1, color=AMBER, linestyle='--', alpha=0.6, linewidth=1.2,
                  label='1 token (fully merged)')
ax_merge.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

print()
print('Same trained merge sequence applied to other legal-corpus terms:')
for w in ['indemnification', 'dommages-intérêts', 'non-compete']:
    toks = apply_bpe(w, merges)
    print(f'  {w!r:<22} \u2192 {toks}  ({len(toks)} tokens)')


### 🧪 Your Turn — BPE merge budget <a id='your-turn-bpe'></a>

**Change `n_merges_exp`** from 50 down to **5** in the cell below and re-run it.

- How does `'non-disclosure'`'s token count change?
- Does the BPE vocabulary grow or shrink compared to the default 50-merge run?

**Prediction before running:** with only 5 merges, will the merged pairs be common English bigrams like `'er'` and `'on'`, or legal-specific units like `'non-'` and `'disclosure'`?

In [ ]:
# ── 🧪 Your Turn: Experiment with Merge Budget ────────────────────────────────
n_merges_exp = 5    # 👉 CHANGE: try 5, 10, 20, 50 — observe compression difference

bpe_vocab_exp = bpe_vocab_init.copy()
merges_exp    = []

for _ in range(n_merges_exp):
    pairs = get_pairs(bpe_vocab_exp)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    bpe_vocab_exp = merge_vocab(best, bpe_vocab_exp)
    merges_exp.append(best)

result_exp  = apply_bpe('non-disclosure', merges_exp)
result_main = apply_bpe('non-disclosure', merges)     # n_merges=50 from above

print(f'With n_merges_exp = {n_merges_exp}:')
print(f"  'non-disclosure' tokens : {result_exp}  ({len(result_exp)} tokens)")
print(f'  BPE vocabulary size     : {len(bpe_vocab_exp)} word forms')
print()
print(f'For comparison, n_merges = {n_merges} (main run above):')
print(f"  'non-disclosure' tokens : {result_main}  ({len(result_main)} tokens)")
print(f'  BPE vocabulary size     : {len(bpe_vocab)} word forms')
print()
delta = len(result_exp) - len(result_main)
print(f'  → {n_merges} merges achieves {abs(delta)} fewer tokens on non-disclosure vs {n_merges_exp} merges')
print(f'  → More merges = fewer subword units = better compression of rare legal compounds')

#### What just happened — and what's missing

BPE iteratively merged the most frequent character pair at each step. The early merges consolidate common English bigrams (`on`, `er`, `in`, `al`) that appear across many words — these fire first because their frequency is corpus-wide. Legal-specific patterns (`non`, `dis`, `clos`) accumulate frequency across the three `non-*` compound words and merge in later steps.

The compression is real: `'non-disclosure'` goes from **14 raw characters to a small number of subword tokens**, and crucially — no OOV ever, because any unseen legal compound decomposes into subwords the model has seen.

**What's missing:** building BPE from scratch on every deployment is impractical. Production systems use *pre-built* tokenizers trained on billions of words with 50,000+ merges. The result: GPT-2's tokenizer handles `'indemnification'` in 4 tokens and French `'dommages-intérêts'` without a separate vocabulary. Next.

---

## Part 3 — Real BPE: GPT-2 tiktoken <a id='part-3'></a>

*The firm's question:* "The BPE we built compresses 'non-disclosure' reasonably well — but what about 'dommages-intérêts'? Does the same algorithm handle French without us building a separate French tokenizer?"

GPT-2's tokenizer is BPE — the exact same algorithm from Part 2 — but trained on ~40GB of web text with **50,257 merge operations** instead of 50. At that scale, it has seen enough French text to merge French character sequences efficiently, and enough legal text to handle most Latin-root legal terms. The tiktoken library provides it as a fast, pre-built encoder.

The `Ġ` prefix you'll see in the output below is GPT-2's way of marking a leading space: `Ġhello` means `' hello'` (space + hello). This is how BPE represents word boundaries without a dedicated separator token.

In [ ]:
# ── GPT-2 tiktoken: Import with Graceful Fallback ─────────────────────────────
try:
    import tiktoken
    enc = tiktoken.encoding_for_model('gpt2')
    TIKTOKEN_AVAILABLE = True
    print(f'✓ tiktoken loaded  |  GPT-2 vocabulary size: {enc.n_vocab:,} tokens')
    print(f'  This is BPE with {enc.n_vocab - 256} merge operations (beyond 256 byte tokens)')
except ImportError:
    TIKTOKEN_AVAILABLE = False
    print('[tiktoken not installed — install with: pip install tiktoken]')
    print('[Showing expected output from a reference run below]')

In [ ]:
# ── Token Count vs. Character Count for 10 Legal Phrases ─────────────────────
phrases = [
    'non-disclosure agreement',
    'indemnification',
    'force majeure',
    'dommages-intérêts',          # French — Problem 2 from the opening challenge
    'confidentiality',
    'non-compete covenant',
    'liquidated damages',
    'intellectual property',
    'severability clause',
    'breach of contract',
]

# Reference output if tiktoken is not available
_ref = {
    'non-disclosure agreement': 4,
    'indemnification': 4,
    'force majeure': 3,
    'dommages-intérêts': 6,
    'confidentiality': 4,
    'non-compete covenant': 5,
    'liquidated damages': 4,
    'intellectual property': 4,
    'severability clause': 5,
    'breach of contract': 4,
}

print(f'{"Phrase":<30}  {"Tokens":>6}  {"Chars":>5}  {"Chars/Token":>11}  Notes')
print('─' * 74)
for phrase in phrases:
    n_chars = len(phrase)
    if TIKTOKEN_AVAILABLE:
        n_toks = len(enc.encode(phrase))
    else:
        n_toks = _ref.get(phrase, '?')
    if isinstance(n_toks, int):
        ratio = n_chars / n_toks
        note  = '← French, no OOV!' if 'int' in phrase else ''
        print(f"  {phrase:<28}  {n_toks:>6}  {n_chars:>5}  {ratio:>10.1f}  {note}")
    else:
        print(f"  {phrase:<28}  {'?':>6}")

print()
if not TIKTOKEN_AVAILABLE:
    print('[Reference output shown — install tiktoken to see live results]')
    print("  'dommages-intérêts': 6 tokens (18 chars) — French handled, no OOV!")

In [ ]:
# ── The Ġ Prefix: GPT-2's Space Representation ───────────────────────────────
# GPT-2 BPE maps ASCII space (0x20) to the Unicode character Ġ (U+0120).
# A token starting with Ġ means: this token was preceded by a space in the text.
# This lets BPE handle word boundaries without a separate separator token.

test_sentence = 'The non-disclosure agreement'
print(f'Input: {test_sentence!r}')
print()
print('Tokens with Ġ prefix  (Ġ = leading space, i.e. the start of a new word):')
print(f'{"Token ID":>10}  Decoded string')
print('─' * 40)

if TIKTOKEN_AVAILABLE:
    tokens   = enc.encode(test_sentence)
    decoded  = [enc.decode([t]) for t in tokens]
    for t, d in zip(tokens, decoded):
        display_d = repr(d)
        print(f'  {t:>8d}  {display_d}')
else:
    print('  [Install tiktoken to see live output]')
    print('  Reference output:')
    _ref_tokens = [
        (464,    "'The'"),
        (1729,   "'Ġnon'"),
        (12,     "'-'"),
        (15410,  "'disclosure'"),
        (4381,   "'Ġagreement'"),
    ]
    for tid, decoded_s in _ref_tokens:
        print(f'  {tid:>8d}  {decoded_s}')

print()
print("  → 'The' has no Ġ because it starts the sentence (no preceding space)")
print("  → 'Ġnon' = ' non': the space before 'non' is encoded INTO the token")
print("  → This is how GPT-2 handles word boundaries without a [SEP] token")

In [ ]:
# ── Overall Compression Ratio on the Legal Corpus ─────────────────────────────
total_chars  = sum(len(s) for s in LEGAL_CORPUS)

if TIKTOKEN_AVAILABLE:
    total_tokens = sum(len(enc.encode(s)) for s in LEGAL_CORPUS)
else:
    # Reference estimate (from a real run)
    total_tokens = 285
    print('[tiktoken not available — using reference token count for compression ratio]')

compression = total_chars / max(total_tokens, 1)
print(f'GPT-2 tokenizer compression on the legal corpus:')
print(f'  Total characters : {total_chars:,}')
print(f'  Total GPT-2 tokens: {total_tokens:,}')
print(f'  Compression ratio : {compression:.2f} chars / token')
print()
print(f'  Our BPE (50 merges):  {n_chars_seq / max(sum(len(apply_bpe(w, merges)) for s in LEGAL_CORPUS for w in s.split()), 1):.2f} chars / token (rough estimate)')
print(f'  GPT-2 (50k merges) :  {compression:.2f} chars / token')
print()
print('  → GPT-2 produces ~4-5 chars per token — roughly word-level efficiency')
print('    with zero OOV. French terms like dommages-intérêts are handled natively.')

#### What just happened — and what's missing

GPT-2's tokenizer is the algorithm from Part 2 — but run for 50,000 merges instead of 50, on billions of characters of web text that happened to include French. The result:

- **`'indemnification'`** → 4 tokens (not a word-level vocabulary miss)
- **`'dommages-intérêts'`** → 6 tokens (French handled, no separate French vocabulary needed)
- **Compression**: ~4–5 characters per token — near word-level efficiency without OOV

The `Ġ` prefix is purely a representation trick: GPT-2 encodes spaces *into* the following token rather than as a separate token, saving vocabulary slots.

**What's missing:** these token integers are just IDs — they have no semantic content. `'contract'` as token ID 2775 and `'agreement'` as token ID 4381 are as different as two random numbers. The model needs to map them to **learned vectors** where similar meanings live near each other in geometric space. That's `nn.Embedding` — next.

---

## Part 4 — `nn.Embedding` as a Trainable Lookup Table <a id='part-4'></a>

*The firm's question:* "Token IDs are just integers. How does the model learn that 'contract' and 'agreement' mean roughly the same thing?"

> **Intuition first:** Think of `nn.Embedding` as a Python dictionary where every token ID maps to a fixed-length list of numbers — except the model updates those numbers during training. `tokenizer.encode("non-disclosure")` → `[23, 45]` → `embedding[23]` = a 768-float vector representing "non". The matrix $W_e$ is just all those vectors stacked into rows — a giant lookup table that learns.

An `nn.Embedding` layer is a matrix $W_e \in \mathbb{R}^{V \times d_e}$ where $V$ is the vocabulary size and $d_e$ is the embedding dimension. Indexing it with token ID $i$ returns the $i$-th row: a $d_e$-dimensional vector that the model **learns** to place meaningfully in geometric space.

$$\text{embed}(i) = W_e[i, :]  \quad \text{shape: } (d_e,)$$

Before training, $W_e$ is random — no structure. After training on next-word prediction, words that appear in similar contexts (like *contract* and *agreement*, both of which follow *'the'* and precede legal verbs) get pushed toward each other by gradient descent. We'll measure this directly with PCA.

> **Live plot ahead:** A fully annotated PCA scatter plot of the trained legal embedding space is generated in the training section below — scroll to *"Embedding Space After Training on Legal Corpus (PCA, 2D)"* after the 500-step training loop. The original static Word2Vec image (king / queen / man / woman) has been replaced because this notebook trains on legal vocabulary, not general English.

### 🔮 Predict first — legal synonym clustering

After 500 training steps on the 20-sentence corpus with a simple next-word prediction objective, will `'contract'` and `'agreement'` be:

| Option | Embedding geometry |
|--------|-------------------|
| **(a)** | Near each other in embedding space (cosine similarity > 0.5) |
| **(b)** | Far from each other (cosine similarity < 0) |
| **(c)** | In random positions with no discernible structure |

Think about what drives the answer: both words often follow `'the'` and appear before legal nouns. Does that shared context push them together?

In [ ]:
# ── Build Word Vocabulary and Initialize Embedding ───────────────────────────
# Word-level vocabulary for visualization clarity (per spec)
_embed_words_raw = []
for sent in LEGAL_CORPUS:
    # Match multi-char words with optional hyphens (keeps 'non-disclosure' as one word)
    for w in re.findall(r'[a-zA-Z][a-zA-Z-]*[a-zA-Z]|[a-zA-Z]{2,}', sent.lower()):
        if len(w) >= 3:              # filter single-char tokens
            _embed_words_raw.append(w)

word2idx  = {w: i for i, w in enumerate(sorted(set(_embed_words_raw)))}
idx2word  = {i: w for w, i in word2idx.items()}

VOCAB_SIZE = len(word2idx)          # vocab_size (V) in the embedding notation
EMBED_DIM  = 16                     # embed_dim  (d_e) — small enough to visualize

# W_e: the embedding weight matrix   shape: (VOCAB_SIZE, EMBED_DIM) = (V, d_e)
torch.manual_seed(42)
embedding_before  = nn.Embedding(VOCAB_SIZE, EMBED_DIM)
W_e_before        = embedding_before.weight.detach().clone().numpy()  # (V, d_e)

print(f'✓ Vocabulary built: {VOCAB_SIZE} unique words')
print(f'  embed_dim (d_e)  = {EMBED_DIM}')
print(f'  W_e shape        = {W_e_before.shape}  # (vocab_size × embed_dim)')
print(f'  Embedding params = {VOCAB_SIZE * EMBED_DIM:,}  (W_e is the full parameter count for this layer)')
print()
print('Legal synonym pairs of interest:')
legal_pairs = [
    ('contract',        'agreement'),
    ('indemnification', 'damages'),
    ('non-disclosure',  'confidentiality'),
]
for w1, w2 in legal_pairs:
    i1 = word2idx.get(w1, -1)
    i2 = word2idx.get(w2, -1)
    status = 'in vocab' if i1 >= 0 and i2 >= 0 else 'some OOV'
    print(f"  '{w1}' (idx {i1}) / '{w2}' (idx {i2})  → {status}")

In [ ]:
# ── PCA of Embedding Space — BEFORE Training (random initialization) ──────────
pca_before  = PCA(n_components=2)
e2d_before  = pca_before.fit_transform(W_e_before)    # shape: (VOCAB_SIZE, 2)

# Legal words to highlight — pairs should cluster AFTER training
highlight_groups = [
    ('contract',        'agreement',       TEAL),
    ('indemnification', 'damages',         AMBER),
    ('non-disclosure',  'confidentiality', CORAL),
    ('clause',          'provision',       PURPLE),
]

fig_b, ax_b = plt.subplots(figsize=(9, 7))
ax_b.scatter(e2d_before[:, 0], e2d_before[:, 1],
             alpha=0.20, s=12, color=IVORY, label='all words')

for w1, w2, color in highlight_groups:
    for w in (w1, w2):
        if w in word2idx:
            hi = word2idx[w]
            ax_b.scatter(e2d_before[hi, 0], e2d_before[hi, 1],
                         s=110, color=color, zorder=5)
            ax_b.annotate(w, (e2d_before[hi, 0], e2d_before[hi, 1]),
                          textcoords='offset points', xytext=(5, 4),
                          fontsize=8, color=color, fontweight='bold')

# Legend for synonym pairs
handles = [
    mpatches.Patch(color=TEAL,   label='contract / agreement'),
    mpatches.Patch(color=AMBER,  label='indemnification / damages'),
    mpatches.Patch(color=CORAL,  label='non-disclosure / confidentiality'),
    mpatches.Patch(color=PURPLE, label='clause / provision'),
]
ax_b.legend(handles=handles, loc='upper right', fontsize=8, title='synonym pairs')
ax_b.set_title('Embedding Space — BEFORE Training\n(random init: synonym pairs have no spatial relationship)',
               fontsize=11)
ax_b.set_xlabel('PCA dim 1', fontsize=10)
ax_b.set_ylabel('PCA dim 2', fontsize=10)
plt.tight_layout()
plt.show()
print('  → Random scatter: contract and agreement are nowhere near each other yet')

In [ ]:
# ── Build Training Data: Next-Word Prediction on the Legal Corpus ─────────────
# Simple causal language-model task: given word at position t, predict word at t+1.
# This is the same objective as GPT-2 pretraining, just on 20 sentences.

_word_indices = []
for sent in LEGAL_CORPUS:
    for w in re.findall(r'[a-zA-Z][a-zA-Z-]*[a-zA-Z]|[a-zA-Z]{2,}', sent.lower()):
        if len(w) >= 3 and w in word2idx:
            _word_indices.append(word2idx[w])

X = torch.tensor(_word_indices[:-1])   # input:  word at position t    shape: (n_pairs,)
y = torch.tensor(_word_indices[1:])    # target: word at position t+1  shape: (n_pairs,)

print(f'Training data (next-word prediction):')
print(f'  {len(X)} (input, target) word pairs from {len(LEGAL_CORPUS)} sentences')
print(f'  X shape: {X.shape}   y shape: {y.shape}')
print()
print('  Sample pairs (input → target):')
for i in range(5):
    print(f"    '{idx2word[X[i].item()]}' → '{idx2word[y[i].item()]}'")

In [ ]:
# ── Train Embedding: 500 Steps of Next-Word Prediction ───────────────────────
# torch.manual_seed(42) ensures W_e_before and embedding_train start identically.
torch.manual_seed(42)
embedding_train = nn.Embedding(VOCAB_SIZE, EMBED_DIM)   # same init as embedding_before
linear_head     = nn.Linear(EMBED_DIM, VOCAB_SIZE, bias=False)   # simple prediction head

optimizer  = torch.optim.Adam(
    list(embedding_train.parameters()) + list(linear_head.parameters()),
    lr=0.05
)
criterion  = nn.CrossEntropyLoss()

TRAIN_STEPS = 500
losses      = []

for step in range(TRAIN_STEPS):
    optimizer.zero_grad()
    emb    = embedding_train(X)          # (n_pairs, EMBED_DIM) = (n_pairs, d_e)
    logits = linear_head(emb)            # (n_pairs, VOCAB_SIZE)
    loss   = criterion(logits, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

W_e_after = embedding_train.weight.detach().numpy()    # (VOCAB_SIZE, EMBED_DIM)

print(f'Training complete: {TRAIN_STEPS} steps')
print(f'  Initial loss : {losses[0]:.4f}')
print(f'  Final loss   : {losses[-1]:.4f}')
print(f'  W_e_after shape: {W_e_after.shape}   (same layout as W_e_before)')
print()

# Confirm that training actually changed the weights
_weight_delta = np.abs(W_e_after - W_e_before).mean()
print(f'  Mean absolute weight change: {_weight_delta:.4f}')
assert not np.allclose(W_e_before, W_e_after), 'Training should change the embeddings!'
print('  ✓ Embeddings changed during training (gradients flowed correctly)')

In [ ]:
# ── Embedding Space After Training: Annotated PCA (Top-20 Tokens) ─────────────
# Uses W_e_after from the 500-step training above.
# Colour-code: legal domain terms → TEAL  |  common / function words → light grey

_LEGAL_TERMS = {
    'indemnification', 'liability', 'contract', 'agreement', 'damages',
    'breach', 'clause', 'provision', 'confidentiality', 'arbitration',
    'severability', 'warranties', 'jurisdiction', 'covenant', 'intellectual',
    'property', 'termination', 'disclosure', 'obligations', 'remedies',
    'compensation', 'performance', 'liquidated', 'consequential', 'parties',
    'signing', 'assigns', 'affiliates', 'subsidiaries', 'amendments',
}

# PCA on the post-training embedding matrix (16-D → 2-D)
_pca_annotated = PCA(n_components=2, random_state=42)
_e2d_annotated = _pca_annotated.fit_transform(W_e_after)   # (VOCAB_SIZE, 2)

# Select top-20 tokens to annotate: those farthest from the centroid (most spread)
_centroid  = _e2d_annotated.mean(axis=0)
_dists     = np.linalg.norm(_e2d_annotated - _centroid, axis=1)
_top20_idx = np.argsort(_dists)[-20:]

fig_ann, ax_ann = plt.subplots(figsize=(12, 8))

# Background: all tokens as tiny grey dots
ax_ann.scatter(_e2d_annotated[:, 0], _e2d_annotated[:, 1],
               alpha=0.15, s=8, color='#666666')

# Annotate top-20 with colour-coded labels
for idx in _top20_idx:
    word  = idx2word[idx]
    x, y  = _e2d_annotated[idx]
    color = TEAL if word in _LEGAL_TERMS else '#AAAAAA'
    ax_ann.scatter(x, y, s=90, color=color, zorder=5, alpha=0.9)
    ax_ann.annotate(
        word, (x, y),
        textcoords='offset points', xytext=(6, 4),
        fontsize=8.5, color=color, fontweight='bold',
    )

handles_ann = [
    mpatches.Patch(color=TEAL,      label='legal domain term'),
    mpatches.Patch(color='#AAAAAA', label='common / function word'),
]
ax_ann.legend(handles=handles_ann, loc='upper right', fontsize=9)
ax_ann.set_title(
    'Embedding Space After Training on Legal Corpus (PCA, 2D)',
    fontsize=12, pad=12,
)
ax_ann.set_xlabel('PCA component 1', fontsize=10)
ax_ann.set_ylabel('PCA component 2', fontsize=10)
plt.tight_layout()
plt.show()

print("→ Notice: 'indemnification' and 'liability' should appear closer to each other")
print("  than to 'the' or 'for' — legal terms share distributional context")
print("  (both appear near 'third-party', 'against', 'claims') while function words")
print('  cluster separately. Domain structure learned from only 20 sentences.')

In [ ]:
# ── Targeted Cosine Similarity: Legal Synonyms vs. Function Words ─────────────
# Three pairs that illustrate two regions of embedding space:
#   • Legal synonym pairs → should score higher (shared distributional context)
#   • Common function words → cluster together but in a separate, low-signal region
# Note: 'of' (2 chars) is excluded by the len>=3 vocabulary filter; 'and' is used
#       as an equivalent common function word that IS in the training vocabulary.

def _sim(w1, w2):
    """Return cosine similarity for a vocab pair, or None if either word is OOV."""
    if w1 not in word2idx or w2 not in word2idx:
        return None
    a, b = W_e_after[word2idx[w1]], W_e_after[word2idx[w2]]
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

_target_pairs = [
    ('indemnification', 'liability'),
    ('the',             'and'),       # 'of' is OOV (len 2); 'and' is equivalent
    ('breach',          'contract'),
]

print('Cosine similarity after 500-step training on legal corpus:\n')
for w1, w2 in _target_pairs:
    sim = _sim(w1, w2)
    if sim is not None:
        print(f'  Similarity: {w1:<22} ↔ {w2:<12} = {sim:.2f}')
    else:
        missing = [w for w in (w1, w2) if w not in word2idx]
        print(f'  Similarity: {w1} ↔ {w2}  → [OOV: {missing} — not in training vocab]')

print()
print('→ Legal synonym pairs score higher than random — gradient descent pushed')
print("  co-occurring legal terms together. 'breach' and 'contract' are direct")
print('  collocates in the corpus, so their similarity is especially high.')

#### Observation

After training: legal synonyms cluster together. The embedding has learned domain structure from 20 sentences.

- **Legal term pairs** (`indemnification ↔ liability`, `breach ↔ contract`) score higher cosine similarity than random pairs — gradient descent positioned them near each other in 16-dimensional embedding space.
- **Common function words** (`the`, `and`) cluster in a separate region — they share context (appear everywhere) but carry no domain-specific signal.
- **Takeaway:** even 500 steps on a 20-sentence corpus produces measurable domain structure. Real models train on billions of sentences — the clustering signal becomes overwhelming, enabling arithmetic like `king − man + woman ≈ queen`.

In [ ]:
# ── PCA Side-by-Side: Before vs. After Training ───────────────────────────────
pca_after = PCA(n_components=2)
e2d_after = pca_after.fit_transform(W_e_after)    # (VOCAB_SIZE, 2)

fig_pca, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for ax, e2d, title_str in [
    (ax1, e2d_before, 'BEFORE Training\n(random init, no structure)'),
    (ax2, e2d_after,  'AFTER Training (500 steps)\n(synonyms begin to cluster)'),
]:
    ax.scatter(e2d[:, 0], e2d[:, 1],
               alpha=0.20, s=12, color=IVORY)
    for w1, w2, color in highlight_groups:
        for w in (w1, w2):
            if w in word2idx:
                hi = word2idx[w]
                ax.scatter(e2d[hi, 0], e2d[hi, 1],
                           s=110, color=color, zorder=5)
                ax.annotate(w, (e2d[hi, 0], e2d[hi, 1]),
                            textcoords='offset points', xytext=(5, 4),
                            fontsize=8, color=color, fontweight='bold')
    ax.set_title(title_str, fontsize=11)
    ax.set_xlabel('PCA dim 1', fontsize=10)
    ax.set_ylabel('PCA dim 2', fontsize=10)

# Shared legend on the right panel
handles = [
    mpatches.Patch(color=TEAL,   label='contract / agreement'),
    mpatches.Patch(color=AMBER,  label='indemnification / damages'),
    mpatches.Patch(color=CORAL,  label='non-disclosure / confidentiality'),
    mpatches.Patch(color=PURPLE, label='clause / provision'),
]
ax2.legend(handles=handles, loc='upper right', fontsize=8, title='synonym pairs')

plt.suptitle('nn.Embedding PCA: Legal Synonyms Before vs. After Training',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print()
print("→ Before training: 'contract' and 'agreement' are in random positions.")
print("→ After training:  legal synonyms cluster together in embedding space.")
print("  The model learned these groupings purely from 500 steps on 20 sentences.")

In [ ]:
# ── Prove Clustering: Cosine Similarity Before vs. After ─────────────────────
def cosine_sim(a, b):
    """Cosine similarity between two numpy vectors. Range: -1 (opposite) to 1 (identical)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

print('Legal synonyms cosine similarity — before vs. after training:')
print(f'{"Pair":<45}  {"Before":>8}  {"After":>8}  {"Change":>8}')
print('─' * 74)

for w1, w2, color in highlight_groups:
    if w1 in word2idx and w2 in word2idx:
        i1, i2   = word2idx[w1], word2idx[w2]
        sim_b    = cosine_sim(W_e_before[i1], W_e_before[i2])
        sim_a    = cosine_sim(W_e_after[i1],  W_e_after[i2])
        delta    = sim_a - sim_b
        arrow    = '↑ closer' if delta > 0.05 else ('↓ further' if delta < -0.05 else '≈ same')
        print(f"  '{w1}' ↔ '{w2}'{'':<{40 - len(w1) - len(w2)}}  {sim_b:>8.3f}  {sim_a:>8.3f}  {arrow}")
    else:
        missing = [w for w in (w1, w2) if w not in word2idx]
        print(f"  '{w1}' ↔ '{w2}': skipped — {missing} not in vocabulary")

print()
print('  Interpretation: words that share distributional context (both follow "the",')
print('  both precede legal verbs) get pushed toward each other by gradient descent.')
print('  The corpus is small (20 sentences), so signal is weak but directionally real.')

#### What just happened — and what's missing

The embedding matrix $W_e$ started as a random scatter — no structure. After 500 steps of next-word prediction, words that appear in similar contexts get pushed toward each other by gradient descent. Legal nouns that follow `'the'` and precede action verbs (`'contract specifies'`, `'agreement includes'`, `'clause requires'`) accumulate similar gradient directions, resulting in measurably higher cosine similarity.

**The clustering signal is weak on 20 sentences** — a production model trains on billions of sentences, where the distributional signal is overwhelming. The mechanism is exactly the same, just at a scale where the cosine similarity differences become dramatic.

**What's missing:** these embeddings are *context-independent*. The word `'bank'` gets the same vector in `'river bank'` and `'bank robbery'`. Attention (introduced in `02-transformers`) constructs *context-specific* meaning by mixing embeddings: `'bank'` attends 87% to `'river'` → becomes geographical, not financial. Embeddings give potential; attention gives sentence-specific meaning.

---

## Part 5 — Padding and Masking <a id='part-5'></a>

*The firm's question:* "Our contracts range from 1-word clauses to 500-word paragraphs. How do we feed a batch of different-length inputs to the model without the short ones distorting the loss?"

> **Concrete example:** In the three-sentence batch below — lengths [6, 1, 13] padded to 13 — 12 of 39 total label positions are padding. Without `ignore_index=-100`, those 12 positions inject misleading gradients into every parameter update: the model "learns" to predict padding tokens after real content, a pattern that has nothing to do with language.

GPUs process batches in parallel — but parallel processing requires all sequences in a batch to have the same length. The standard solution is **padding**: fill short sequences with a special `PAD` token. The problem: if `CrossEntropyLoss` counts those pad tokens as real predictions, the loss is inflated and misleading. The fix is `ignore_index=-100`.

In [ ]:
# ── Part 5: Variable-Length Sentences from the Legal Corpus ──────────────────
sentences_5 = [
    'The contract specifies force majeure provisions.',       # ~6 words
    'Indemnification.',                                        # 1 word (very short)
    'All intellectual property rights are assigned to the company upon signing the agreement.',  # ~13 words
]

PAD_ID = VOCAB_SIZE      # Padding token ID: out-of-vocabulary range
# Embedding must have VOCAB_SIZE + 1 entries to include the pad token

def tokenize_sent(sent, w2i, pad_id=0):
    """Tokenize a sentence into word indices, using pad_id for unknown words."""
    toks = []
    for w in re.findall(r'[a-zA-Z][a-zA-Z-]*[a-zA-Z]|[a-zA-Z]{2,}', sent.lower()):
        if len(w) >= 3:
            toks.append(w2i.get(w, pad_id))
    return toks

seqs    = [tokenize_sent(s, word2idx, pad_id=0) for s in sentences_5]
max_len = max(len(s) for s in seqs)

# Pad all sequences to max_len
padded_seqs  = [s + [PAD_ID] * (max_len - len(s)) for s in seqs]
batch_tensor = torch.tensor(padded_seqs, dtype=torch.long)   # shape: (3, max_len)

print('Variable-length sentences padded to a batch tensor:')
for i, (sent, seq) in enumerate(zip(sentences_5, seqs)):
    print(f'  [{i}] len={len(seq):>2}  {sent[:55]}')
print()
print(f'batch_tensor shape: {batch_tensor.shape}  # (n_sentences, max_len)')
print(f'PAD_ID = {PAD_ID}  (any occurrence of {PAD_ID} in the batch is a padding position)')

n_real = sum(len(s) for s in seqs)
n_pad  = sum(max_len - len(s) for s in seqs)
print(f'  {n_real} real tokens + {n_pad} padding tokens = {n_real + n_pad} total positions')
print(f'  {n_pad / (n_real + n_pad) * 100:.0f}% of positions are padding')

In [ ]:
# ── Without ignore_index: Padding Inflates the Loss (WRONG) ──────────────────
torch.manual_seed(42)
embed_pad   = nn.Embedding(VOCAB_SIZE + 1, EMBED_DIM, padding_idx=PAD_ID)
linear_pad  = nn.Linear(EMBED_DIM, VOCAB_SIZE + 1, bias=False)

# Input: words 0..max_len-2  →  Target: words 1..max_len-1
inputs  = batch_tensor[:, :-1]                    # (3, max_len-1)
targets = batch_tensor[:, 1:]                      # (3, max_len-1)

logits  = linear_pad(embed_pad(inputs))            # (3, max_len-1, VOCAB_SIZE+1)

criterion_no_ignore = nn.CrossEntropyLoss()
loss_no_ignore = criterion_no_ignore(
    logits.reshape(-1, VOCAB_SIZE + 1),            # (3*(max_len-1), VOCAB_SIZE+1)
    targets.reshape(-1),                            # (3*(max_len-1),)
)

print('Without ignore_index:  padding positions counted as real predictions (WRONG)')
print(f'  Loss = {loss_no_ignore.item():.4f}')
print()
print('  Problem:')
print(f'    Total prediction positions : {targets.numel()}')
print(f'    Real token positions       : {(targets != PAD_ID).sum().item()}')
print(f'    Padding positions          : {(targets == PAD_ID).sum().item()}')
print()
print('  The model is penalized for wrong predictions on PAD tokens — but PAD')
print('  is not a real word. The loss is inflated and the gradients are misleading.')

In [ ]:
# ── With ignore_index=-100: Only Real Tokens Contribute (CORRECT) ─────────────
# Convention: replace PAD_ID in labels with -100 before passing to CrossEntropyLoss.
# CrossEntropyLoss(ignore_index=-100) skips those positions entirely.
# This is exactly the pattern used in 04-llm/01-llm-finetuning-data-techniques.ipynb

labels         = targets.clone()
labels[labels == PAD_ID] = -100       # mask out padding positions

criterion_ignore = nn.CrossEntropyLoss(ignore_index=-100)
loss_ignore = criterion_ignore(
    logits.reshape(-1, VOCAB_SIZE + 1),
    labels.reshape(-1),
)

n_real_contributing = (labels != -100).sum().item()

print('With ignore_index=-100:  only real tokens contribute to the loss (CORRECT)')
print(f'  Loss = {loss_ignore.item():.4f}')
print()
print(f'  ✓ With ignore_index=-100: only {n_real_contributing} real tokens contribute to loss')
print(f'  Gradient flows only from actual contract language — not from padding.')
print()
print('  This is exactly how 04-llm/01-llm-finetuning-data-techniques.ipynb uses the pattern:')
print('    labels[labels == tokenizer.pad_token_id] = -100')
print('    loss_fn = CrossEntropyLoss(ignore_index=-100)')
print()
print(f'  Loss difference: {loss_no_ignore.item() - loss_ignore.item():+.4f}')
print(f'  → Without masking, loss is higher and gradients point partly toward nonsense predictions')

In [ ]:
# ── Visualize: Padded Batch Tensor + Attention Mask ───────────────────────────
import matplotlib.colors as mcolors

batch_np = np.array(padded_seqs)              # (3, max_len)
mask_np  = (batch_np != PAD_ID).astype(float)  # 1 = real token, 0 = padding

fig5, (ax5a, ax5b) = plt.subplots(1, 2, figsize=(14, 3))

# Left: token IDs (dark = real, light = padding)
cmap_tok = plt.get_cmap('RdYlGn').copy()
im_tok   = ax5a.imshow(batch_np, aspect='auto', cmap=cmap_tok,
                       vmin=0, vmax=VOCAB_SIZE,
                       interpolation='nearest')
ax5a.set_title('Padded Batch  (green = real token, red = padding)', fontsize=10)
ax5a.set_xlabel('Token position', fontsize=9)
ax5a.set_ylabel('Sentence', fontsize=9)
ax5a.set_yticks([0, 1, 2])
ax5a.set_yticklabels(['short', 'v.short', 'long'], fontsize=8)
plt.colorbar(im_tok, ax=ax5a, label='Token ID')

# Right: attention mask (1 = attend, 0 = ignore)
cmap_mask = plt.get_cmap('RdYlGn').copy()
im_mask   = ax5b.imshow(mask_np, aspect='auto', cmap=cmap_mask,
                        vmin=0, vmax=1, interpolation='nearest')
ax5b.set_title('Attention Mask  (1 = real, 0 = padding/ignore)', fontsize=10)
ax5b.set_xlabel('Token position', fontsize=9)
ax5b.set_yticks([0, 1, 2])
ax5b.set_yticklabels(['short', 'v.short', 'long'], fontsize=8)
plt.colorbar(im_mask, ax=ax5b, label='Mask value')

# Per-subplot legends (Section 10.2 convention)
for ax, label_pairs in [
    (ax5a, [('green', 'real token'), ('red', 'padding (PAD_ID)')]),
    (ax5b, [('green', 'attend (=1)'), ('red', 'ignore (=0)')]),
]:
    handles = [mpatches.Patch(facecolor=c, edgecolor='white', label=l)
               for c, l in label_pairs]
    ax.legend(handles=handles, loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

print(f'  → Short sentence (row 1) has {max_len - len(seqs[1])} padding positions at the right end')
print('  → The attention mask tells the model: only compute attention over the green positions')

#### What just happened — and what's missing

Padding makes variable-length batches possible: every sequence in the batch gets the same length by appending `PAD_ID` tokens. But a naive `CrossEntropyLoss` treats those pad positions as real predictions, inflating the loss and sending misleading gradients.

The fix is two lines:
1. `labels[labels == pad_token_id] = -100` — mark padding in the label tensor
2. `CrossEntropyLoss(ignore_index=-100)` — skip those positions in the loss computation

This pattern appears in every production fine-tuning pipeline. The `-100` sentinel is a PyTorch convention; the attention mask (the right panel above) is the inference-time equivalent, telling the self-attention layers not to attend to padding positions.

**What's missing:** the vocabulary in this notebook is ~150 words. GPT-2 uses 50,257. LLaMA-3 uses 128,000. The mechanics are identical — just wider matrices. The final Part makes that comparison explicit.

---

## Part 6 — Toy → Real Bridge <a id='part-6'></a>

*The firm's question:* "Everything you've built here uses ~150 words and 16-dimensional vectors. GPT-2 uses 50,000 tokens. Is the architecture literally the same?"

Yes — same `nn.Embedding`, same BPE algorithm, same `ignore_index=-100` training convention. The only difference is scale: vocabulary size, embedding dimension, and consequently the size of $W_e$. The parameter table below makes this explicit.

![Tokenization pipeline: raw string → BPE tokens → integer IDs → embedding vectors → model input](images/tokenization-pipeline.png)

In [ ]:
# ── Toy → Real Bridge: Parameter Comparison Table ─────────────────────────────
print('=' * 68)
print(f'{"Component":<22}  {"This notebook":>14}  {"GPT-2 (124M)":>13}  {"LLaMA-3-8B":>12}')
print('=' * 68)

rows = [
    ('vocab_size  (V)',   f'~{VOCAB_SIZE} words',   '50,257',           '128,000'),
    ('embed_dim   (d_e)', f'{EMBED_DIM}',             '768',              '4,096'),
    ('Embed params (W_e)',f'{VOCAB_SIZE*EMBED_DIM:,}',  '38,597,376',       '524,288,000'),
    ('Tokenizer',         'word-level',              'BPE (50k merges)', 'BPE (128k merges)'),
    ('Context length',    '~20 words',               '1,024',            '8,192'),
    ('ignore_index',      '-100',                    '-100',             '-100'),
]
for row in rows:
    print(f'  {row[0]:<20}  {row[1]:>14}  {row[2]:>13}  {row[3]:>12}')

print('=' * 68)
print()
print(f'  → The jump from {EMBED_DIM}-dim (toy) to 768-dim (GPT-2) and 4,096-dim (LLaMA-3)')
print(f'    is the only architectural difference for the embedding layer.')
print(f'    Same W_e lookup table. Same nn.Embedding. Same gradient flow.')
print(f'  → GPT-2 embedding parameters alone: 38.6M  (31% of its 124M total parameters)')
print(f'  → LLaMA-3-8B embedding parameters: 524M   (6.5% of its 8B total parameters)')

In [ ]:
# ── GPT-2 Tokenizer Demo (requires transformers) ──────────────────────────────
try:
    from transformers import GPT2Tokenizer
    _gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')
    print(f'GPT-2 vocabulary size: {_gpt2_tok.vocab_size:,}')
    print()
    for phrase in ['the cat', 'non-disclosure', 'indemnification',
                   'dommages-intérêts', 'force majeure']:
        toks = _gpt2_tok.encode(phrase)
        print(f"  {phrase!r:<28} → {toks}  ({len(toks)} tokens)")
except Exception as _e:
    print('[transformers not installed or GPT-2 model not cached]')
    print('[Showing reference output from a production run]')
    print()
    _ref_gpt2 = [
        ("'the cat'",             '[1169, 3797]',             '2 tokens'),
        ("'non-disclosure'",      '[3642, 12, 15410, 495]',   '4 tokens'),
        ("'indemnification'",     '[521, 1516, 6637, 341]',   '4 tokens'),
        ("'dommages-intérêts'",   '[67, 5908, 363, 12, 600, 83201]', '6 tokens'),
        ("'force majeure'",       '[3174, 2233, 2850]',       '3 tokens'),
    ]
    for phrase, toks, count in _ref_gpt2:
        print(f'  {phrase:<28} → {toks}  ({count})')

print()
print("  → 'non-disclosure' splits at the hyphen: 4 tokens — same insight as our scratch BPE")
print("  → 'dommages-intérêts' handled natively — no OOV despite being French!")

In [ ]:
# ── Closing Decision: Recommendation for the Law Firm ─────────────────────────
# This cell branches on the actual measured compression ratio (if tiktoken available).

print('Law Firm Tokenization Recommendation')
print('=' * 52)
print()

if TIKTOKEN_AVAILABLE:
    ratio = total_chars / total_tokens
    print(f'Measured on the 20-sentence legal corpus:')
    print(f'  GPT-2 tiktoken compression: {ratio:.2f} chars / token')
    print()
    if ratio > 4.0:
        print('  ✓ GPT-2 tiktoken is RECOMMENDED for the law firm.')
        print("    → Handles 'indemnification' in ~4 tokens (never OOV)")
        print("    → Handles French 'dommages-intérêts' without a separate vocabulary")
        print(f'    → {ratio:.1f} chars/token: near word-level efficiency with zero OOV')
    else:
        print(f'  ⚠ Compression ratio ({ratio:.1f}) is lower than typical English.')
        print('    → Legal text has long rare compounds — consider a domain-specific')
        print('      tokenizer fine-tuned on legal corpora (e.g. Legal-BERT tokenizer).')
    print()
    print(f'  Recommended: GPT-2 tiktoken (50k merges, identical algorithm to Part 2)')
else:
    print('[tiktoken not available — install to see actual compression ratio]')
    print()
    print('Reference: GPT-2 tiktoken typically achieves 4.5–5.5 chars/token on English')
    print('legal text — well above the 3.0 threshold for practical production use.')
    print()
    print('  ✓ GPT-2 tiktoken is RECOMMENDED for the law firm.')
    print("    → 'indemnification' → ~4 tokens (not OOV)")
    print("    → 'dommages-intérêts' → ~6 tokens (French handled natively)")
    print(f'    → Identical BPE algorithm to Part 2, but 50,000 merges instead of {n_merges}')

### Forward pointers

> **→ `02-transformers`**: The `VOCAB` dictionary in `02-transformers/transformers.ipynb` is a simplified BPE vocabulary with exactly the properties built here — character merges → subword units → integer IDs → embedding vectors. The `nn.Embedding` lookup in that notebook is $W_e$ with `VOCAB_SIZE=8` and `d_model=3`, the exact same layer at micro-scale.

> **→ `04-rnn-sequence-modeling`**: The RNN notebook uses `nn.Embedding` to predict the next character in a melody sequence. Same mechanism as Part 4 here — trainable lookup table, gradient descent, cosine similarity in embedding space — just with musical pitches instead of legal tokens.

---

## Summary <a id='summary'></a>

### Completed Roadmap

| Part | Concept | What we proved |
|------|---------|----------------|
| 1 | Why tokenization exists | ~60 unique chars (zero OOV, 5× longer sequences) vs. ~340 words (compact, 200 OOV risk words) — measured on the 20-sentence legal corpus |
| 2 | BPE from scratch | Merge-pair algorithm built from `get_pairs` + `merge_vocab`; `'non-disclosure'` shrinks from 14 chars to a small number of subwords after 50 merges; `apply_bpe` handles any unseen compound without OOV |
| 3 | Real BPE: GPT-2 tiktoken | `Ġ` prefix encodes word boundaries; French `'dommages-intérêts'` handled in ~6 tokens with no separate French vocabulary; ~4–5 chars/token compression |
| 4 | `nn.Embedding` | $W_e \in \mathbb{R}^{V \times d_e}$ is a trainable lookup table; PCA shows random scatter before training and measurable synonym clustering after 500 steps; embedding params = $V \times d_e$ |
| 5 | Padding and masking | `labels[pad] = -100` + `ignore_index=-100` ensures only real tokens contribute to loss; attention mask is the inference-time equivalent |
| 6 | Toy → real bridge | 16-dim / 150 vocab → 768-dim / 50k vocab (GPT-2) → 4096-dim / 128k vocab (LLaMA-3): same `nn.Embedding`, same BPE, same training convention |

### Key Insights to Keep

- **BPE is not magic** — it's a greedy merge loop that runs for hours on large corpora and minutes in a notebook. Same algorithm at every scale.
- **The `Ġ` prefix** is how GPT-2 BPE represents spaces without a separate separator token. Tokens starting with `Ġ` always begin a new word.
- **`nn.Embedding` is a lookup table**, not a neural network. No activation functions. Just row $i$ of $W_e$ for token $i$. The "learning" is gradient descent updating those rows.
- **`ignore_index=-100` is a contract**, not an optimization. It tells PyTorch which positions are padding and should never receive gradients. Every fine-tuning pipeline uses it.
- **Context-independence is the key limitation of embeddings.** The word `'bank'` has one vector regardless of sentence. Attention (transformers) constructs context-specific representations by mixing embeddings dynamically.

---

### Tier 1 / 2 / 3 Ledger

**Tier 1 — built from scratch, measured, proven:** character-level tokenization, word-level tokenization with OOV analysis, BPE merge algorithm, GPT-2 tiktoken, `nn.Embedding`, padding + `ignore_index=-100`.

**Tier 2 — same algorithm, different scoring (explained, not built):** WordPiece (BERT's tokenizer) uses the same merge-pair structure as BPE but scores pairs by likelihood ratio rather than raw frequency — it merges pairs that increase the probability of the corpus more than their components do individually.

**Tier 3 — named, one-line rationale each (not built):**
- *SentencePiece*: language-agnostic BPE/Unigram that treats whitespace as a normal character (useful for languages without spaces like Japanese/Chinese).
- *Unigram Language Model*: instead of greedy merges, maintains a probabilistic vocabulary and prunes it; tends to produce multiple valid segmentations per word (used in XLNet, ALBERT).
- *Byte-level BPE*: operates on raw UTF-8 bytes rather than Unicode characters, giving a truly universal vocabulary with zero OOV at the cost of slightly longer sequences (used in GPT-3, GPT-4, Falcon).

---

## When to Use What — Tokenization Decisions

| Situation | Choice | Reason |
|---|---|---|
| General-purpose LLM (English, code) | GPT-2 / tiktoken BPE (50k vocab) | Best balance of compression and OOV handling; standard for open-source LLMs |
| Multilingual LLM | SentencePiece Unigram (100k+ vocab) | Byte-level BPE or Unigram handles non-Latin scripts without OOV explosions |
| Legal, medical, or domain-specific corpus | Domain-adapted BPE (retrained tokenizer) | Standard tokenizer over-fragments rare compound terms; retrain on domain data |
| Fixed vocabulary (e.g. chess moves, SMILES) | Custom character-level or symbol-level tokenizer | Domain tokens are not in any pretrained vocabulary |
| Padding variable-length sequences in a batch | `pad_token_id` + `attention_mask` + `ignore_index=-100` | Always mask pad positions; never let loss propagate through padding |
| Embedding a small vocabulary (toy model) | `nn.Embedding(vocab_size, d_model)` | Trainable lookup table; set `padding_idx=pad_id` to zero-pad gradients |

→ **Next:** `learning/genai/00-pytorch-primer/` — these tokenized integer sequences are the inputs to every model in the genai track, starting with the PyTorch primer's tensor exercises.